![image_1788767485114.png](./image_1788767485114.png "image_1788767485114.png")

- **Schema evolution mode:** `addNewColumnsWithTypeWidening`, not `addNewColumns` or `rescue` –
  dataset has both a new column and an oversized numeric value, this mode handles both
  (new column + int->long widening) instead of routing the value to `_rescued_data`.


In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbr_dev_ua5816bd.roksolana_shendiu770_bronze.petroleum_consumption_bronze (
    record_id STRING
)
TBLPROPERTIES ('delta.enableTypeWidening' = 'true')

In [0]:
from pyspark.sql.functions import col

CATALOG = "dbr_dev_ua5816bd"
SCHEMA_LANDING = "roksolana_shendiu770"
SCHEMA_BRONZE = "roksolana_shendiu770_bronze"

SOURCE_PATH = f"/Volumes/{CATALOG}/{SCHEMA_LANDING}/bronze_landing/petroleum_consumption"
SCHEMA_LOCATION = f"/Volumes/{CATALOG}/{SCHEMA_LANDING}/bronze_landing/_schemas/petroleum_consumption"
CHECKPOINT_LOCATION = f"/Volumes/{CATALOG}/{SCHEMA_LANDING}/bronze_landing/_checkpoints/petroleum_consumption"
TARGET_TABLE = f"{CATALOG}.{SCHEMA_BRONZE}.petroleum_consumption_bronze"

raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumnsWithTypeWidening")
    .option("cloudFiles.maxFilesPerTrigger", 100)
    .load(SOURCE_PATH)
    .select("*", col("_metadata.file_path").alias("source_file_path"))
)

query = (
    raw_stream.writeStream
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE)
)

query.awaitTermination()

![image_1788770724325.png](./image_1788770724325.png "image_1788770724325.png")

**Expected behavior:** stream failed with `UnknownFieldException` after encountering
`price_per_barrel`. Auto Loader first updated the schema in `schemaLocation`, then
stopped the stream – restart resumes with the new column already in place.

![image_1788770832249.png](./image_1788770832249.png "image_1788770832249.png")